# Warm-Up: The 2x2 DiD, Five Ways

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bdepro/courses/blob/main/eco4400/warmup.ipynb)

ECO 4400 &middot; Applied Research & Case Analytics &middot; Journal Replication, Stage 1

Full instructions (what to read first, what to turn in) are on the [task page](https://bdepro.github.io/courses/eco4400/replication-warmup.html). This notebook is where you do the work.

**Before you run anything:** read Scott Cunningham's *"Diff-in-diff can be written down six ways!"* (Scott's Mixtape). He shows six equivalent ways to the same DiD number — two manual calculations, four regressions. This notebook builds five of them: both manual calculations, plus three of the four regressions.

Run each cell top to bottom (Shift+Enter). All five numbers below should match.

## Setup

Load the data straight from a public URL — no download, no upload, nothing to configure. `castle.dta` is state-level homicide data around the 2006 adoption of "castle doctrine" self-defense laws (Cheng & Hoekstra), hosted publicly by Scott Cunningham.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_stata("https://github.com/scunning1975/mixtape/raw/master/castle.dta")
df.head()

## Build the 2x2

Restrict to a clean 2x2: states that adopted castle doctrine in 2006 (treatment) vs. states that hadn't adopted by the end of the sample window (control), comparing 2005 to 2006.

In [ ]:
# Drop states whose law took effect in a year other than 2006 -- keeps a
# clean treatment (2006 adopters) vs. control (never-adopters) comparison.
df = df[~df["effyear"].isin([2005, 2007, 2008, 2009])].copy()

df["post"] = (df["year"] >= 2006).astype(int)
df["treat"] = (df["effyear"] == 2006).astype(int)

df = df[df["year"].isin([2005, 2006])].copy()
print(df.shape)

## 1. Manual 2x2, first differences then difference

Compute the average `l_homicide` in each of the four `treat×post` cells, then `(y11 − y10) − (y01 − y00)`. This number is your target for every method below.

In [ ]:
y11 = df.loc[(df.treat == 1) & (df.post == 1), "l_homicide"].mean()
y10 = df.loc[(df.treat == 1) & (df.post == 0), "l_homicide"].mean()
y01 = df.loc[(df.treat == 0) & (df.post == 1), "l_homicide"].mean()
y00 = df.loc[(df.treat == 0) & (df.post == 0), "l_homicide"].mean()

did_manual = (y11 - y10) - (y01 - y00)
print(f"1. Manual 2x2 (first differences, then difference):   {did_manual:.6f}")

## 2. Manual 2x2, group differences then difference

Same four averages as Method 1, subtracted in the other order: `(y11 − y01) − (y10 − y00)`. The signs move with the terms, so the number doesn't change — only the order you subtract in does.

In [ ]:
did_manual2 = (y11 - y01) - (y10 - y00)
print(f"2. Manual 2x2 (group differences, then difference):   {did_manual2:.6f}")

## 3. OLS, treatment × post interaction

Regress `l_homicide` on `treat`, `post`, and their interaction. The coefficient on the interaction term is your DiD estimate — the same OLS estimator as every regression below, just with different right-hand-side variables.

In [ ]:
m1 = smf.ols("l_homicide ~ post * treat", data=df).fit(cov_type="HC3")
print(f"3. OLS, treatment x post interaction:                  {m1.params['post:treat']:.6f}")

## 4. Twoway fixed effects

Regress `l_homicide` on the `treat×post` interaction, controlling for a dummy for every state and a dummy for year. Still OLS — just with many more right-hand-side variables. The interaction coefficient should still match your target.

In [ ]:
m2 = smf.ols("l_homicide ~ treat:post + C(sid) + C(year)", data=df).fit(cov_type="HC3")
print(f"4. Twoway fixed effects:                               {m2.params['treat:post']:.6f}")

## 5. First-difference regression

For each state, compute the change in `l_homicide` from 2005 to 2006, then regress that change on `treat`. Same coefficient again.

In [ ]:
df_sorted = df.sort_values(["sid", "year"])
df_sorted["diff"] = df_sorted.groupby("sid")["l_homicide"].diff()
change_df = df_sorted[df_sorted["year"] == 2006]

m3 = smf.ols("diff ~ treat", data=change_df).fit(cov_type="HC3")
print(f"5. First-difference regression:                        {m3.params['treat']:.6f}")

All five numbers above should match. That is the point: diff-in-diff is a 2x2 comparison first, and a regression only incidentally.

**Bonus (optional):** Cunningham also shows a sixth way — a fourth regression, of the treat-minus-control gap on a post dummy. It's not built here on purpose: it makes the same point Method 2 already makes (the order of subtraction doesn't matter), just as a regression instead of by hand. If you want to build it yourself as a check, it's a three-line addition to the cell above.

## Results table (bonus)

The same five numbers, formatted as a three-line, journal-style table — the way you'd see it in an actual paper.

In [ ]:
def fmt(value, width=12, decimals=3):
    if value is None:
        return f"{'-':>{width}}"
    if isinstance(value, str):
        return f"{value:>{width}}"
    return f"{value:>{width}.{decimals}f}"


def fmt_se(value, width=12, decimals=3):
    if value is None:
        return f"{'-':>{width}}"
    return f"{'(' + format(value, f'.{decimals}f') + ')':>{width}}"


LABEL_WIDTH = 20
COL_WIDTH = 12
TABLE_WIDTH = LABEL_WIDTH + COL_WIDTH * 5
RULE = "=" * TABLE_WIDTH
THIN_RULE = "-" * TABLE_WIDTH

print(RULE)
print("TABLE 1. THE EFFECT OF CASTLE DOCTRINE LAWS ON HOMICIDES".center(TABLE_WIDTH))
print("Five Equivalent Ways to the Same Estimate".center(TABLE_WIDTH))
print(RULE)
print("Dependent variable: Log homicide rate")
print(THIN_RULE)

print(" " * LABEL_WIDTH + "".join(fmt(f"({i})", COL_WIDTH) for i in range(1, 6)))
spec_line1 = ["Manual,", "Manual,", "OLS", "Twoway", "1st-Diff."]
spec_line2 = ["1st Diffs", "Grp Diffs", "Interaction", "FE", "Regression"]
print(" " * LABEL_WIDTH + "".join(fmt(s, COL_WIDTH) for s in spec_line1))
print(" " * LABEL_WIDTH + "".join(fmt(s, COL_WIDTH) for s in spec_line2))
print(THIN_RULE)

coefs = [did_manual, did_manual2, m1.params["post:treat"], m2.params["treat:post"], m3.params["treat"]]
print(f"{'Treat x Post':<{LABEL_WIDTH}}" + "".join(fmt(c, COL_WIDTH) for c in coefs))

ses = [None, None, m1.bse["post:treat"], m2.bse["treat:post"], m3.bse["treat"]]
print(f"{'':<{LABEL_WIDTH}}" + "".join(fmt_se(s, COL_WIDTH) for s in ses))
print()

state_fe = [None, None, None, "Yes", None]
year_fe = [None, None, None, "Yes", None]
print(f"{'State FE':<{LABEL_WIDTH}}" + "".join(fmt(x, COL_WIDTH) for x in state_fe))
print(f"{'Year FE':<{LABEL_WIDTH}}" + "".join(fmt(x, COL_WIDTH) for x in year_fe))
print(THIN_RULE)

n_obs = [len(df), len(df), len(df), len(df), len(change_df)]
r2 = [None, None, m1.rsquared, m2.rsquared, m3.rsquared]
print(f"{'Observations':<{LABEL_WIDTH}}" + "".join(fmt(n, COL_WIDTH, decimals=0) for n in n_obs))
print(f"{'R-squared':<{LABEL_WIDTH}}" + "".join(fmt(r, COL_WIDTH) for r in r2))
print(RULE)
print("Notes: Robust (HC3) standard errors in parentheses. Sample is 42")
print("states observed in 2005 and 2006 (Cheng and Hoekstra castle-doctrine")
print("data, via Cunningham 2021). Columns (1)-(2) are manual calculations,")
print("not regressions. Column (3) is OLS with a treatment x post")
print("interaction. Column (4) adds state and year fixed effects. Column")
print("(5) collapses to one observation per state (its 2005-2006 change in")
print("the outcome) and regresses that change on treatment status.")
print(RULE)

## Your answer

In 3&ndash;4 sentences: *why* do all five methods have to produce the same number? What does that tell you about what a DiD estimate actually is?

*Double-click this cell to write your answer here.*